<a href="https://colab.research.google.com/github/tekpinar/deepflex/blob/main/flexibilitiesFromRosettaFoldModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---

# **Calculate Protein Flexibility from RosettaFold Models**

---
Attention: If you run some cells multiple times, we highly recommend you to clean all files and restart! You can go to the dropdown menu 'Disconnect and delete runtime' (under the Runtime menu) for a complete fresh restart.


# Install all required libraries

In [ ]:
!pip install py3dmol
!pip install biopython

# Upload your pdb file containing five models, which you downloaded from Robetta server.

In [ ]:
from google.colab import files
import os

print("Please upload your multimodel PDB file:")
uploaded = files.upload()

# Get the first uploaded file name
if uploaded:
    fn = list(uploaded.keys())[0]
    print(f"File '{fn}' uploaded successfully.")
else:
    print("No file uploaded.")
    fn = None

# Create some functions that will be used later

In [ ]:
from scipy.stats import rankdata,zscore

def rankSortData(dataArray):
    """
        This function ranksorts protein data and converts it
        to values between [0,1.0].

    Parameters:
    ----------
    dataArray: numpy array of arrays
               data read by numpy.loadtxt

    Returns:
    -------
    normalizedRankedDataArray: numpy array
    """
    normalizedRankedDataArray = rankdata(dataArray)/float(len(dataArray))

    return (normalizedRankedDataArray)

def get_bfactor_range_v2(pdb_file):
    """
    Extract B-factor values from PDB file and return their range

    Parameters:
    pdb_file (str): Path to the PDB file

    Returns:
    tuple: (minimum B-factor, maximum B-factor)
    """
    bfactors = []
    # print(pdb_file[0])
    allLines = pdb_file[0].split('\n')
    for line in allLines:
      # print(line)
      if line.startswith('ATOM') or line.startswith('HETATM'):
          try:
            # B-factor is typically in columns 61-66
            bfactor = float(line[60:66].strip())
            bfactors.append(bfactor)
          except (ValueError, IndexError):
            continue

    if not bfactors:
      return (0, 100)  # default range if no B-factors found

    return (min(bfactors), max(bfactors))

def visualize_protein_bfactor(pdb_file):
    """
    Visualize protein structure in cartoon representation colored by B-factor
    using automatically determined range and rainbow colors

    Parameters:
    pdb_file (str): Path to the PDB file
    """

    # Create a py3Dmol view instance
    view = py3Dmol.view()

    # Get B-factor range from the file
    bfactor_min, bfactor_max = get_bfactor_range_v2(pdb_file)

    # # Load the PDB file
    # with open(pdb_file, 'r') as f:
    #     pdb_data = f.read()

    # # Add the molecule to the viewer
    # view.addModel(pdb_data, "pdb")

    for pdb_string in pdb_file:
      view.addModel(pdb_string, 'pdb')

    # Set cartoon representation with rainbow coloring based on B-factor
    view.setStyle({'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'linear',  # Using rainbow color scheme
            'min': bfactor_min,
            'max': bfactor_max,
            'colors': ["blue", "white", "red"]
        }
    }})

    # Center and zoom the view
    view.zoomTo()

    # Add legend for B-factor coloring
    view.addPropertyLabels(
        prop='b',
        gradient='bwr',
        min=bfactor_min,
        max=bfactor_max,
        legend={'x': 0.85, 'y': 0.5}
    )

    # # Add text showing the B-factor range
    # view.addLabel(f"B-factor range: {bfactor_min:.2f} - {bfactor_max:.2f}",
    #              {'position': {'x': -20, 'y': -20, 'z': 0},
    #               'backgroundColor': 'white',
    #               'fontColor': 'black'})

    return view


# Calculate MSF from RosettaFold models


In [ ]:
from Bio.PDB import PDBParser, Selection, Superimposer
import numpy as np
import os
import pandas as pd
import sys
import glob

rosettafold_structure_dir = './'
print(f"rosettafold structure directory: {rosettafold_structure_dir}")
rosettafold_msf_data = []
parser = PDBParser(QUIET=True)

# Get a list of all protein IDs from the NMR data to ensure consistency
protein_id = 'test_protein'

protein_id_lower = protein_id.lower() # Ensure case consistency
protein_id_upper = protein_id.upper()
print(f"Processing rosettafold structures for protein: {protein_id}")
pdb_files = []

# pdb_file_path = os.path.join(specific_fold_path, f'robetta_models_*.pdb')
pdb_files = glob.glob(rosettafold_structure_dir+'/*.pdb')
pdb_file = pdb_files[0] if pdb_files else None
print('pdb_file:', pdb_file)
try:
    structure_id = pdb_file # e.g., '2lfi'
    # full_path=os.path.join(main_path, pdb_file, pdb_file+'.pdb')
    print(f"Processing {structure_id}")
    # structure = parser.get_structure(pdb_file, os.path.join(main_path, 'nmr_structures', pdb_file, pdb_file+'.pdb'))
    structure = parser.get_structure(protein_id, pdb_file)

    # Extract models and C-alpha atoms
    models = list(structure.get_models())
    if not models:
        print(f"Warning: No models found in {pdb_file}. Skipping.")
        sys.exit(-1)

    # Get C-alpha atoms for all models
    all_ca_atoms = []
    for model in models:
        ca_atoms = [atom for atom in Selection.unfold_entities(model, 'A') if atom.get_id() == 'CA']
        all_ca_atoms.append(ca_atoms)

    if not all_ca_atoms or not all_ca_atoms[0]:
        print(f"Warning: No C-alpha atoms found in {pdb_file}. Skipping.")
        sys.exit(-1)

    # Ensure all models have the same number of C-alpha atoms for alignment
    num_ca_per_model = [len(ca_list) for ca_list in all_ca_atoms]
    if not all(n == num_ca_per_model[0] for n in num_ca_per_model):
        print(f"Warning: Inconsistent number of C-alpha atoms across models in {pdb_file}. Skipping.")
        sys.exit(-1)

    num_residues = num_ca_per_model[0]

    # Structural alignment to the first model
    superimposer = Superimposer()
    mobile_coords = []
    fixed_coords = np.array([atom.get_coord() for atom in all_ca_atoms[0]])

    for i, ca_atoms_model in enumerate(all_ca_atoms):
        if i == 0: # First model is the fixed reference
            mobile_coords.append(fixed_coords)
            continue

        mobile_atoms = ca_atoms_model
        fixed_atoms = all_ca_atoms[0]

        # Map atoms for superimposition
        fixed_atoms_map = []
        mobile_atoms_map = []
        for fixed_atom, mobile_atom in zip(fixed_atoms, mobile_atoms):
            # Check if residue numbers and chain IDs match, as a basic sanity check
            if fixed_atom.get_parent().get_id() == mobile_atom.get_parent().get_id() and \
                fixed_atom.get_parent().get_parent().get_id() == mobile_atom.get_parent().get_parent().get_id():
                fixed_atoms_map.append(fixed_atom)
                mobile_atoms_map.append(mobile_atom)

        if len(fixed_atoms_map) != num_residues or len(mobile_atoms_map) != num_residues:
            print(f"Warning: Mismatch in C-alpha atom count for alignment in {pdb_file} model {i+1}. Skipping alignment for this model.")
            mobile_coords.append(np.array([atom.get_coord() for atom in ca_atoms_model])) # Use unaligned if issue
        else:
            superimposer.set_atoms(fixed_atoms_map, mobile_atoms_map)
            superimposer.apply(mobile_atoms)
            mobile_coords.append(np.array([atom.get_coord() for atom in mobile_atoms]))

    mobile_coords = np.array(mobile_coords)

    # Calculate MSF
    # Mean coordinates across all models for each residue
    mean_coords = np.mean(mobile_coords, axis=0)
    # Squared fluctuations for each residue across all models
    sq_fluctuations = np.sum(np.square(mobile_coords - mean_coords), axis=(0, 2)) / (len(models) - 1)

    # Ranksort normalize MSF values
    if len(sq_fluctuations) > 1:
        normalized_msf = rankSortData(sq_fluctuations)
    else:
        normalized_msf = sq_fluctuations # If only one value, no normalization needed

    # Store results
    residue_numbers = [atom.get_parent().get_id()[1] for atom in all_ca_atoms[0]]
    res_idx = 0
    for res_num, msf_val in zip(residue_numbers, normalized_msf):
        res_idx += 1
        rosettafold_msf_data.append({
            'protein_id': protein_id,
            'residue_number': res_num,
            'residue_index': res_idx,
            'normalized_msf': msf_val,
            'source': 'RosettaFold'
        })

except Exception as e:
    print(f"Error processing {pdb_file}: {e}")


rosettafold_msf_df = pd.DataFrame(rosettafold_msf_data)
#print(f"Processed MSF data for {len(rosettafold_msf_df['protein_id'].unique())} unique rosettafold proteins.")
#print("rosettafold MSF Data (first 5 rows):")
print(rosettafold_msf_df.head())

# Assign normalized MSF to Bfactors columns, align models and write a pdb file for each model

In [ ]:
print(pdb_file)

# 1. Create a dictionary to map residue_index to normalized_msf values
msf_map = {}
if 'rosettafold_msf_data' in globals() and isinstance(rosettafold_msf_data, list):
    if not rosettafold_msf_data:
        print("Warning: `rosettafold_msf_data` is an empty list. All PDB B-factors will be set to 0.0.")
    else:
        for entry in rosettafold_msf_data:
            if 'residue_index' in entry and 'normalized_msf' in entry:
                msf_map[entry['residue_index']] = entry['normalized_msf']
            else:
                print(f"Warning: Entry in alphafold3_msf_data missing 'residue_index' or 'normalized_msf': {entry}")
else:
    print("Warning: `rosettafold_msf_data` is not defined or is not a list. All PDB B-factors will be set to 0.0.")

# Align all 5 pdb files to the first model from row["pdb_id"]}_{experimentalMethod}_msf.pdb using only calpha atoms.
# Save the aligned models in a multimodel pdb file named {row["pdb_id"]}_RosettaFold_msf.pdb in the output directory.
# Use bio.PDB module to do this. Use the first model from row["pdb_id"]}_{experimentalMethod}_msf.pdb as the reference structure, and align all 5 cif files to this reference structure. Use only calpha atoms for the alignment. Save the aligned models in a multimodel pdb file named {row["pdb_id"]}_RosettaFold_msf.pdb in the output directory.
from Bio import PDB
from Bio.PDB import PDBIO
import io
parser = PDB.PDBParser(QUIET=True)
io_object = PDB.PDBIO()

ref_structure = parser.get_structure(pdb_file.split('.')[1], pdb_file)
ref_model = ref_structure[0]
ref_atoms = [atom for atom in ref_model.get_atoms() if atom.get_id() == 'CA']
aligned_models = []
structure = parser.get_structure(pdb_file.split('.')[1], pdb_file)
aligned_pdb_strings = []
for i in range(5): # 0 to 4
    print(f'Aligning Model {i+1} to reference model')
    model = structure[i]
    atoms = [atom for atom in model.get_atoms() if atom.get_id() == 'CA']
    super_imposer = PDB.Superimposer()
    super_imposer.set_atoms(ref_atoms, atoms)
    super_imposer.apply(model.get_atoms())
    aligned_models.append(model)

    for chain in model:
      for residue in chain:
          # Get the PDB residue number (e.g., from (' ', 1, ' ')) as the key
          # This matches the 'residue_index' if they are sequential and 1-based.
          res_num = residue.get_id()[1]

          # Look up the corresponding normalized_msf value from msf_map
          msf_value = msf_map.get(res_num, 0.0)
          if res_num not in msf_map:
              # Only print warning if msf_map actually has data but this specific res_num is missing
              if msf_map:
                  print(f"Info: MSF value for residue index {res_num} not found in map for {pdb_file}. Setting B-factor to 0.0.")

          # Assign this normalized_msf value to the B-factor of *all* atoms in the residue
          # Biopython expects B-factors to be floats.
          for atom in residue.get_atoms():
              atom.set_bfactor(float(msf_value))
    io_object = PDB.PDBIO()
    buffer = io.StringIO()
    io_object.set_structure(model)
    io_object.save(buffer)
    aligned_pdb_strings.append(buffer.getvalue())
    with open(f'model{i+1}.pdb', 'w') as f:
        # io_object.set_structure(model)
        io_object.save(f, write_end=False)
        f.write('END\n')


# Verify creation of the pdb files

In [ ]:
import glob

# List all .pdb files to confirm their creation
pdb_files = ['model1.pdb', 'model2.pdb', 'model3.pdb', 'model4.pdb', 'model5.pdb']
print("Newly created PDB files:")
for pdb_file in pdb_files:
    print(pdb_file)

if len(pdb_files) == 5:
    print(f"Confirmation: Successfully created {len(pdb_files)} PDB files.")
else:
    print(f"Warning: Expected 5 PDB files, but found {len(pdb_files)}.")

## Visualize the aligned models with py3Dmol







In [ ]:
import py3Dmol
import pandas as pd
import re
from IPython.display import display

# Replace with your PDB file path
view = visualize_protein_bfactor(aligned_pdb_strings)
view.show()

# Plot 'Residue Numbers vs Normalized MSF' as an interactive 2D plot.

In [ ]:
import plotly.express as px

if 'rosettafold_msf_df' in globals() and not rosettafold_msf_df.empty:
    fig = px.line(
        rosettafold_msf_df,
        x='residue_number',
        y='normalized_msf',
        title='Normalized Mean Squared Fluctuation (MSF) vs. Residue Number',
        labels={'residue_number': 'Residue Number', 'normalized_msf': 'Normalized MSF'},
        hover_data={'residue_number': True, 'normalized_msf': ':.2f'}
    )
    fig.update_traces(mode='lines+markers', marker=dict(size=5))
    fig.update_layout(xaxis_title='Residue Number', yaxis_title='Normalized MSF')
    fig.show()
else:
    print("Error: rosettafold_msf_df is not available or is empty. Cannot generate plot.")

# Download the models with flexibilities at Bfactors as a zip file

In [ ]:
import zipfile
from google.colab import files
import glob
import os

# Identify all newly created PDB files
pdb_files = ['model1.pdb', 'model2.pdb', 'model3.pdb', 'model4.pdb', 'model5.pdb']

# Define the name for the output zip file
zip_filename = 'rosettafold_flexibility_models.zip'

# Create a zip file and add all PDB files to it
with zipfile.ZipFile(zip_filename, 'w') as zf:
    for pdb_file in pdb_files:
        zf.write(pdb_file, os.path.basename(pdb_file))

print(f"Successfully created '{zip_filename}' containing {len(pdb_files)} PDB models.")

# Provide the download link to the user
files.download(zip_filename)